In [ ]:
# Updated Garbage Classification Model with Enhanced Fusion, Training, and Evaluation Strategies

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, datasets, models
from torch.utils.data import DataLoader, WeightedRandomSampler, Subset
from torchvision.datasets import ImageFolder
from torchvision.models.feature_extraction import create_feature_extractor
import numpy as np
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import StratifiedShuffleSplit

# Config
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_path = "/content/garbage_classification/garbage_classification"
NUM_CLASSES = 10
BATCH_SIZE = 32
EPOCHS = 15
LEARNING_RATE = 1e-4
MIXUP_ALPHA = 0.4
LABEL_SMOOTHING = 0.1

# MixUp functions
def mixup_data(x, y, alpha=MIXUP_ALPHA):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    index = torch.randperm(x.size(0)).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# Data transforms
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.2),
    transforms.RandomRotation(15),
    transforms.RandomAffine(10, translate=(0.05, 0.05)),
    transforms.RandomPerspective(0.3, p=0.5),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Load dataset
full_dataset = ImageFolder(root=data_path)
labels = [label for _, label in full_dataset]
splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, temp_idx = next(splitter.split(np.zeros(len(labels)), labels))
temp_labels = [labels[i] for i in temp_idx]
val_split = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
val_idx, test_idx = next(val_split.split(np.zeros(len(temp_labels)), temp_labels))
val_idx = [temp_idx[i] for i in val_idx]
test_idx = [temp_idx[i] for i in test_idx]

class DatasetWithTransform(torch.utils.data.Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, idx):
        x, y = self.subset[idx]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)

train_dataset = DatasetWithTransform(Subset(full_dataset, train_idx), train_transform)
val_dataset = DatasetWithTransform(Subset(full_dataset, val_idx), val_transform)
test_dataset = DatasetWithTransform(Subset(full_dataset, test_idx), val_transform)

# Weighted sampling
class_counts = [0] * NUM_CLASSES
for _, label in train_dataset:
    class_counts[label] += 1
class_weights = 1. / torch.tensor(class_counts, dtype=torch.float)
samples_weights = [class_weights[label] for _, label in train_dataset]
sampler = WeightedRandomSampler(samples_weights, len(samples_weights), replacement=True)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# SE block
class SEBlock(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction),
            nn.ReLU(),
            nn.Linear(in_channels // reduction, in_channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

# Feature extractors

def create_multi_level_resnet():
    return create_feature_extractor(
        models.resnet50(weights="DEFAULT"),
        return_nodes={"layer2": "feat2", "layer3": "feat3", "layer4": "feat4"}
    )

def create_multi_level_swin():
    swin = models.swin_t(weights="DEFAULT")
    return create_feature_extractor(swin, return_nodes={"flatten": "feat"})

class BiFusionLayer(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=dim, nhead=4, batch_first=True), num_layers=1)

    def forward(self, x):
        return self.transformer(x.unsqueeze(1)).squeeze(1)

class GatedMultiHeadAttentionFusion(nn.Module):
    def __init__(self, dim=768, heads=4, context_dim=16):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.context_generator = nn.Sequential(
            nn.Linear(dim, 64), nn.ReLU(), nn.Linear(64, context_dim))
        self.gate = nn.Sequential(
            nn.Linear(2 * dim + context_dim, 128),
            nn.ReLU(),
            nn.Linear(128, heads + 1),
            nn.Softmax(dim=1))
        self.norm = nn.LayerNorm(dim)

    def forward(self, res_feat, swin_feat):
        res_feat = res_feat.unsqueeze(1)
        swin_feat = swin_feat.unsqueeze(1)
        combined = torch.cat([res_feat, swin_feat], dim=1)
        attn_output, _ = self.cross_attn(combined, combined, combined)
        attn_avg = self.norm(attn_output.mean(dim=1))
        context = self.context_generator(attn_avg)
        gate_input = torch.cat([res_feat.squeeze(1), swin_feat.squeeze(1), context], dim=1)
        gates = self.gate(gate_input)
        head_gates = gates[:, :-1].unsqueeze(-1)
        context_gate = gates[:, -1].unsqueeze(-1)
        fused = attn_avg * context_gate + (res_feat.squeeze(1) + swin_feat.squeeze(1)) * head_gates.mean(dim=1)
        return fused

class AttentionFusionClassifier(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.resnet = create_multi_level_resnet()
        self.swin = create_multi_level_swin()
        self.se2 = SEBlock(512)
        self.se3 = SEBlock(1024)
        self.se4 = SEBlock(2048)
        self.proj_res2 = nn.Linear(512, 768)
        self.proj_res3 = nn.Linear(1024, 768)
        self.proj_res4 = nn.Linear(2048, 768)
        self.proj_swin = nn.Linear(768, 768)
        self.res_weight = nn.Parameter(torch.ones(3))
        self.bifusion = BiFusionLayer(768)
        self.fusion = GatedMultiHeadAttentionFusion()
        self.classifier = nn.Sequential(
            nn.Linear(768, 256), nn.LayerNorm(256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, num_classes))
        self.aux_res_classifier = nn.Linear(768, num_classes)
        self.aux_swin_classifier = nn.Linear(768, num_classes)

    def forward(self, x):
        res_feats = self.resnet(x)
        res2 = self.proj_res2(self.se2(res_feats['feat2']).mean(dim=[2, 3]))
        res3 = self.proj_res3(self.se3(res_feats['feat3']).mean(dim=[2, 3]))
        res4 = self.proj_res4(self.se4(res_feats['feat4']).mean(dim=[2, 3]))
        w = F.softmax(self.res_weight, dim=0)
        res_feat = self.bifusion(w[0]*res2 + w[1]*res3 + w[2]*res4)
        swin_feat = self.swin(x)['feat']
        swin_feat = swin_feat.mean(dim=[2, 3]) if swin_feat.dim() == 4 else swin_feat.mean(dim=1) if swin_feat.dim() == 3 else swin_feat
        swin_feat = self.bifusion(self.proj_swin(swin_feat))
        fused = self.fusion(res_feat, swin_feat)
        main_out = self.classifier(fused)
        aux_res_out = self.aux_res_classifier(res_feat)
        aux_swin_out = self.aux_swin_classifier(swin_feat)
        return main_out, aux_res_out, aux_swin_out, fused

# Training
class AuxLossWeightScheduler:
    def __init__(self, total_epochs):
        self.total_epochs = total_epochs
    def get_weight(self, epoch):
        return max(0.0, 1 - epoch / self.total_epochs)

def train_one_epoch(model, loader, optimizer, criterion, epoch, loss_scheduler):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(inputs)
        main_out, aux1, aux2, _ = outputs
        loss_main = criterion(main_out, labels)
        loss_aux1 = criterion(aux1, labels)
        loss_aux2 = criterion(aux2, labels)
        lambda_aux = loss_scheduler.get_weight(epoch)
        loss = loss_main + lambda_aux * (loss_aux1 + loss_aux2)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(main_out, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return running_loss / len(loader.dataset), correct / total

def evaluate(model, loader, criterion):
    model.eval()
    correct, total, loss_total = 0, 0, 0.0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = model(inputs)[0]
            loss = criterion(outputs, labels)
            loss_total += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return loss_total / len(loader.dataset), correct / total

def train_model(model, train_loader, val_loader, epochs):
    model.to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
    loss_scheduler = AuxLossWeightScheduler(epochs)
    best_val_acc = 0
    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, epoch, loss_scheduler)
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        scheduler.step()
        print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), "best_model.pth")
            print("Best model saved.")

def evaluate_full_metrics(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = model(inputs)[0]
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average='macro')
    rec = recall_score(all_labels, all_preds, average='macro')
    f1 = f1_score(all_labels, all_preds, average='macro')
    cm = confusion_matrix(all_labels, all_preds)
    return acc, prec, rec, f1, cm

# Main
if __name__ == "__main__":
    model = AttentionFusionClassifier()
    train_model(model, train_loader, val_loader, EPOCHS)
    model.load_state_dict(torch.load("best_model.pth"))
    acc, prec, rec, f1, cm = evaluate_full_metrics(model, test_loader)
    print("\n--- Test Metrics ---")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print("Confusion Matrix:\n", cm)


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 200MB/s]
Downloading: "https://download.pytorch.org/models/swin_t-704ceda3.pth" to /root/.cache/torch/hub/checkpoints/swin_t-704ceda3.pth
100%|██████████| 108M/108M [00:00<00:00, 163MB/s] 


Epoch 1 | Train Loss: 2.9344 | Train Acc: 0.8261 | Val Loss: 0.6166 | Val Acc: 0.9580
Best model saved.
Epoch 2 | Train Loss: 2.2406 | Train Acc: 0.9042 | Val Loss: 0.5910 | Val Acc: 0.9636
Best model saved.
Epoch 3 | Train Loss: 1.9629 | Train Acc: 0.9278 | Val Loss: 0.5775 | Val Acc: 0.9687
Best model saved.
Epoch 4 | Train Loss: 1.8095 | Train Acc: 0.9389 | Val Loss: 0.5739 | Val Acc: 0.9711
Best model saved.
Epoch 5 | Train Loss: 1.6727 | Train Acc: 0.9414 | Val Loss: 0.5656 | Val Acc: 0.9729
Best model saved.
Epoch 6 | Train Loss: 1.5540 | Train Acc: 0.9471 | Val Loss: 0.5633 | Val Acc: 0.9762
Best model saved.
Epoch 7 | Train Loss: 1.4060 | Train Acc: 0.9556 | Val Loss: 0.5651 | Val Acc: 0.9739
Epoch 8 | Train Loss: 1.3080 | Train Acc: 0.9563 | Val Loss: 0.5584 | Val Acc: 0.9757
Epoch 9 | Train Loss: 1.1770 | Train Acc: 0.9648 | Val Loss: 0.5524 | Val Acc: 0.9799
Best model saved.
Epoch 10 | Train Loss: 1.0773 | Train Acc: 0.9692 | Val Loss: 0.5542 | Val Acc: 0.9790
Epoch 11 | Tr